# Example 1.1.2 / 3: In the Model or in the Data?

Companion notebook for **1.1.2 Set Declaration vs. Definition** in the AMPL lab Documentation, section "In the Model or in the Data?". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## In the Model or in the Data?

| The set ... | Define it in the model (`=`) | Give it in the data (`:=`) |
|---|---|---|
| is part of the model's logic (shift types, a fixed range) | yes | |
| is derived from other sets or parameters | yes | |
| changes from one data set to the next | | yes |
| is large (hundreds of members or more) | | yes |

The four forms from the top of this page fit together as one decision:


Only a declared set, with or without `default`, can later change through data, `let` or Python.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SHIFTS = {"Day", "Evening", "Night"};   # structural: fixed in the model
set PROD;                                    # instance data: from the data
set CITIES;                                  # instance data: from the data

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["CITIES"] = ["GARY", "CLEV", "PITT"]

In [5]:
# run.py
for name in ["SHIFTS", "PROD", "CITIES"]:
    print(name, ampl.set[name].to_list())

SHIFTS ['Day', 'Evening', 'Night']
PROD ['bands', 'coils', 'plate']
CITIES ['GARY', 'CLEV', 'PITT']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set CITIES := GARY CLEV PITT;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display SHIFTS, PROD, CITIES;

set SHIFTS := Day Evening Night;

set PROD := bands coils plate;

set CITIES := GARY CLEV PITT;

